In [10]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="2"

In [11]:
from setproctitle import setproctitle
setproctitle("ddqn_vs_ppo")

In [12]:
import sys
sys.path.append('..')

In [13]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu, True)

In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import numpy as np
from tqdm import trange
from UltimateTicTacToeEnvSelfPlay import UltimateTicTacToeEnvSelfPlay
from DoubleDQNAgent import DoubleDQNAgent
from PPOAgent_with_improvements import PPOAgent

In [16]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
ddqn_agent = DoubleDQNAgent(action_space_size, state_space_shape, loaded=True, model_path="../models/Thesis_DDQN_data_augmentation_2mln")
ppo_agent = PPOAgent(state_space_shape, action_space_size, loaded=True, model_path="../models/Thesis_PPO_zero_entropy")
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    ddqn_states = np.array([env.to_state()[0] for env in envs])
    ddqn_available_actions = np.array([env.to_state()[1] for env in envs])
    ddqn_actions = ddqn_agent.choose_action(ddqn_states, ddqn_available_actions, True)
    ddqn_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, ddqn_reward[i], game_finished[i], _  = envs[i].step(ddqn_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = ddqn_reward[i]
    
    ppo_states = np.array([env.to_state()[0] for env in envs])
    ppo_available_actions = np.array([env.to_state()[1] for env in envs])
    ppo_actions = ppo_agent.act(ppo_states, ppo_available_actions)
    ppo_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, ppo_reward[i], game_finished[i], _  = envs[i].step(ppo_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -ppo_reward[i]
    print("Both players have done a move.")

win_as_first_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_first_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as first player: ", win_as_first_player)
print("draw rate as first player: ", draw_as_first_player)

Models loaded from memory
Models loaded from memory
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
2  out of  500
Both players have done a move.
7  out of  500
Both players have done a move.
18  out of  500
Both players have done a move.
43  out of  500
Both players have done a move.
70  out of  500
Both players have done a move.
110  out of  500
Both players have done a move.
185  out of  500
Both players have done a move.
257  out of  500
Both

In [17]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
ddqn_agent = DoubleDQNAgent(action_space_size, state_space_shape, loaded=True, model_path="../models/Thesis_DDQN_data_augmentation_2mln")
ppo_agent = PPOAgent(state_space_shape, action_space_size, loaded=True, model_path="../models/Thesis_PPO_zero_entropy")
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    ppo_states = np.array([env.to_state()[0] for env in envs])
    ppo_available_actions = np.array([env.to_state()[1] for env in envs])
    ppo_actions = ppo_agent.act(ppo_states, ppo_available_actions)
    ppo_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, ppo_reward[i], game_finished[i], _  = envs[i].step(ppo_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -ppo_reward[i]
                
    ddqn_states = np.array([env.to_state()[0] for env in envs])
    ddqn_available_actions = np.array([env.to_state()[1] for env in envs])
    ddqn_actions = ddqn_agent.choose_action(ddqn_states, ddqn_available_actions, True)
    dqn_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, dqn_reward[i], game_finished[i], _  = envs[i].step(ddqn_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = dqn_reward[i]
    print("Both players have done a move.")

win_as_second_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_second_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as second player: ", win_as_second_player)
print("draw rate as second player: ", draw_as_second_player)

Models loaded from memory
Models loaded from memory
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
3  out of  500
Both players have done a move.
6  out of  500
Both players have done a move.
15  out of  500
Both players have done a move.
38  out of  500
Both players have done a move.
88  out of  500
Both players have done a move.
142  out of  500
Both players have done a move.
213  out of  500
Both p

In [18]:
total_win = (win_as_first_player+win_as_second_player)/2
total_draw = (draw_as_first_player+draw_as_second_player)/2
print("Total win rate: ", total_win)
print("Total draw rate: ", total_draw)
print("Total loss: ", 1-total_draw-total_win)

Total win rate:  0.881
Total draw rate:  0.01
Total loss:  0.10899999999999999
